# NB_Bronze_Load
Converts one day of raw JSON pages into the Delta table `bronze.<resource>`.

* Bronze is **append-only history**: every row delivered by the API is kept (including repeats). Deduplication and change detection happen in Silver.
* The full nested FHIR resource is stored in the struct column `resource` (aligned to the existing table schema); no flattening happens in Bronze.
* Metadata columns: `extraction_timestamp` (when the API page was called), `api_url_or_params`, `source_file`, `ingest_date`, `run_id`.
* Re-running is safe: a Raw run that is already fully loaded is skipped; a partial load stops the notebook for investigation.

In [ ]:
# Parameters (injected by the pipeline)
resource_name = "Condition"   # Patient / Encounter / Observation / Condition
load_date     = "2026-11-03"  # YYYY-MM-DD
run_id        = ""

In [ ]:
# ============================================================================================================
# CELL 1: BRONZE CONFIGURATION # Patient / Encounter / Observation / Condition
# =============================================================================================================


from datetime import datetime, timezone
from pyspark.sql import functions as F, Window
from delta.tables import DeltaTable

target = f"bronze.{resource_name.lower()}"

raw_base = (
    f"Files/raw/{resource_name}/ingest_date={load_date}"
)

now_utc = lambda: datetime.now(timezone.utc).replace(tzinfo=None)

print("Resource:", resource_name)
print("Load date:", load_date)
print("Target table:", target)
print("Raw path:", raw_base)

In [59]:
# ============================================
# CELL 2: PIPELINE LOG SCHEMA
# ============================================

LOG_SCHEMA = (
    "run_id string, resource_name string, load_date string, "
    "layer string, step string, page_no int, api_url string, "
    "api_call_time timestamp, http_status int, rows_fetched int, "
    "rows_written long, api_total long, raw_file_path string, "
    "data_saved_time timestamp, status string, message string"
)

spark.table("meta.pipeline_log").printSchema()

print("Logging schema configured.")

StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 61, Finished, Available, Finished, False)

root
 |-- run_id: string (nullable = true)
 |-- resource_name: string (nullable = true)
 |-- layer: string (nullable = true)
 |-- page_no: integer (nullable = true)
 |-- api_url: string (nullable = true)
 |-- api_call_time: timestamp (nullable = true)
 |-- http_status: integer (nullable = true)
 |-- rows_fetched: integer (nullable = true)
 |-- raw_file_path: string (nullable = true)
 |-- data_saved_time: timestamp (nullable = true)
 |-- status: string (nullable = true)
 |-- error_message: string (nullable = true)
 |-- load_date: string (nullable = true)
 |-- step: string (nullable = true)
 |-- rows_written: long (nullable = true)
 |-- api_total: long (nullable = true)

Logging schema configured.


In [60]:
# ============================================
# CELL 3: LOGGING FUNCTIONS
# ============================================

def log_rows(rows):

    df_log = spark.createDataFrame(rows, LOG_SCHEMA)

    df_log = df_log.select(
        F.col("run_id").cast("string"),
        F.col("resource_name").cast("string"),
        F.col("layer").cast("string"),
        F.col("page_no").cast("int"),
        F.col("api_url").cast("string"),
        F.col("api_call_time").cast("timestamp"),
        F.col("http_status").cast("int"),
        F.col("rows_fetched").cast("int"),
        F.col("raw_file_path").cast("string"),
        F.col("data_saved_time").cast("timestamp"),
        F.col("status").cast("string"),
        F.col("message").cast("string").alias("error_message"),
        F.col("load_date").cast("string"),
        F.col("step").cast("string"),
        F.col("rows_written").cast("long"),
        F.col("api_total").cast("long")
    )

    df_log.write.format("delta").mode("append").saveAsTable(
        "meta.pipeline_log"
    )


def log_summary(
    layer,
    fetched,
    written,
    status="SUCCESS",
    message=None,
    api_total=None,
    api_url=None,
    path=None
):

    log_rows([(
        run_id,
        resource_name,
        load_date,
        layer,
        "summary",
        None,
        api_url,
        None,
        None,
        fetched,
        written,
        api_total,
        path,
        now_utc(),
        status,
        message
    )])


print("Logging functions created successfully.")

StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 62, Finished, Available, Finished, False)

Logging functions created successfully.


In [61]:
# ============================================
# CELL 4: FIND RAW INGESTION RUN
# ============================================

raw_summary = (
    spark.table("meta.pipeline_log")
    .filter(
        (F.col("resource_name") == resource_name) &
        (F.col("load_date") == load_date) &
        (F.col("layer") == "raw") &
        (F.col("step") == "summary") &
        (F.col("status").isin("SUCCESS", "WARN"))
    )
    .orderBy(F.col("data_saved_time").desc())
)

display(
    raw_summary.select(
        "run_id",
        "resource_name",
        "load_date",
        "status",
        "rows_fetched",
        "rows_written",
        "raw_file_path",
        "data_saved_time"
    )
)

StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 63, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3ee6edb9-8d93-411e-97ba-952496e75503)

In [62]:
# ============================================
# CELL 5: SELECT EXACT RAW RUN
# ============================================

raw_record = raw_summary.first()

if raw_record is None:
    raise RuntimeError("No Raw ingestion run found.")

run_id = raw_record["run_id"]
raw_rows = raw_record["rows_written"]

if not run_id or raw_rows is None:
    raise RuntimeError("Raw run ID or row count is missing.")

raw_dir = f"{raw_base}/run_id={run_id}"

print("Selected Raw Run ID:", run_id)
print("Expected Raw Records:", raw_rows)
print("Raw Directory:", raw_dir)

print("Bronze Target:", target)

StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 64, Finished, Available, Finished, False)

Selected Raw Run ID: 660f0af9-ac1a-48f0-812b-989c7c3ba0c1
Expected Raw Records: 989
Raw Directory: Files/raw/Condition/ingest_date=2026-11-03/run_id=660f0af9-ac1a-48f0-812b-989c7c3ba0c1
Bronze Target: bronze.condition


In [63]:
# ============================================
# CELL 6: VALIDATE RAW FILES
# ============================================

raw_check = (
    spark.read
    .option("multiLine", "true")
    .json(raw_dir)
)

raw_check = raw_check.withColumn(
    "source_file",
    F.input_file_name()
)

print("Raw JSON files:", 
      raw_check.select("source_file").distinct().count())

print("FHIR bundle count:", raw_check.count())

print("Raw schema:")
raw_check.printSchema()

display(
    raw_check.select("source_file").distinct()
)

StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 65, Finished, Available, Finished, False)

Raw JSON files: 10
FHIR bundle count: 10
Raw schema:
root
 |-- entry: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- fullUrl: string (nullable = true)
 |    |    |-- resource: struct (nullable = true)
 |    |    |    |-- abatementDateTime: string (nullable = true)
 |    |    |    |-- asserter: struct (nullable = true)
 |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- reference: string (nullable = true)
 |    |    |    |-- bodySite: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullab

SynapseWidget(Synapse.DataFrame, 4431f1ce-e5f6-4352-9548-70e328a18e2c)

In [64]:

# ============================================
# CELL 7: EXTRACT FHIR RESOURCES
# ============================================

df = (
    raw_check
    .select(
        "source_file",
        F.explode("entry").alias("e")
    )
    .filter(
        F.col("e.resource.resourceType") == resource_name
    )
    .select(
        F.col("e.resource").alias("resource"),
        F.col("e.resource.id").alias("resource_id"),
        F.col("e.resource.meta.lastUpdated")
            .cast("string").alias("source_last_updated"),
        "source_file"
    )
)

print(f"Extracted {resource_name} records:", df.count())

print(
    f"Distinct {resource_name} IDs:",
    df.select("resource_id").distinct().count()
)

print(
    f"Missing {resource_name} IDs:",
    df.filter(
        F.col("resource_id").isNull()
    ).count()
)

display(
    df.select(
        "resource_id",
        "source_last_updated",
        "source_file"
    ).limit(10)
)


StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 66, Finished, Available, Finished, False)

Extracted Condition records: 989
Distinct Condition IDs: 989
Missing Condition IDs: 0


SynapseWidget(Synapse.DataFrame, 1c11f8ba-ac30-439c-9111-17c2db99b7f5)

In [65]:
# ============================================
# CELL 8: CHECK EXISTING BRONZE TABLE
# ============================================

if spark.catalog.tableExists(target):

    print("Bronze table already exists:", target)

    bronze_existing = spark.table(target)

    print("Existing Bronze schema:")
    bronze_existing.printSchema()

    print("Existing Bronze rows:",
          bronze_existing.count())

    print("Incoming columns:")
    print(df.columns)

else:
    print("Bronze table does not exist:", target)
    print("A new Bronze table will be created.")

StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 67, Finished, Available, Finished, False)

Bronze table already exists: bronze.condition
Existing Bronze schema:
root
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- bodySite: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |-- text: string (nullable = true)
 |    |-- category: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |-- system: strin

In [ ]:
# ============================================
# CELL 9: ALIGN EXISTING RESOURCE SCHEMA
# ============================================

if spark.catalog.tableExists(target):

    existing_schema = spark.table(target).schema

    if "resource" not in existing_schema.fieldNames():
        raise RuntimeError(
            "Existing Bronze table has no resource column."
        )

    resource_schema = existing_schema["resource"].dataType

    df = df.withColumn(
        "resource",
        F.from_json(
            F.to_json(F.col("resource")),
            resource_schema
        )
    )

print("Resource schema aligned.")

df.printSchema()

In [ ]:
# ============================================
# CELL 10: GET RAW PAGE TIMESTAMPS
# ============================================

pages = (
    spark.table("meta.pipeline_log")
    .filter(
        (F.col("run_id") == run_id) &
        (F.col("resource_name") == resource_name) &
        (F.col("load_date") == load_date) &
        (F.col("layer") == "raw") &
        (F.col("step") == "page") &
        (F.col("status") == "SUCCESS")
    )
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("raw_file_path")
            .orderBy(F.col("data_saved_time").desc())
        )
    )
    .filter(F.col("rn") == 1)
    .select(
        "raw_file_path",
        F.col("api_call_time").alias("page_call_time"),
        F.col("api_url").alias("page_api_url")
    )
)

print("Successful Raw pages:", pages.count())

display(pages)

In [ ]:
# ============================================
# CELL 11: ATTACH INGESTION METADATA
# ============================================

df = (
    df.join(
        F.broadcast(pages),
        F.expr(
            "source_file LIKE concat('%', raw_file_path)"
        ),
        "left"
    )
    .withColumn(
        "extraction_timestamp",
        F.coalesce(
            F.col("page_call_time"),
            F.current_timestamp()
        )
    )
    .withColumn(
        "api_url_or_params",
        F.col("page_api_url")
    )
    .withColumn(
        "ingest_date",
        F.lit(load_date)
    )
    .withColumn(
        "run_id",
        F.lit(run_id)
    )
    .select(
        "resource",
        "resource_id",
        "source_last_updated",
        "api_url_or_params",
        "source_file",
        "extraction_timestamp",
        "ingest_date",
        "run_id"
    )
)

print("Records with metadata:", df.count())

print(
    "Missing extraction timestamps:",
    df.filter(
        F.col("extraction_timestamp").isNull()
    ).count()
)

display(
    df.select(
        "resource_id",
        "extraction_timestamp",
        "ingest_date",
        "run_id"
    ).limit(10)
)

In [69]:

# ============================================
# CELL 12: VALIDATE BRONZE INPUT
# ============================================

n_in = df.count()

missing_timestamps = (
    df.filter(
        F.col("extraction_timestamp").isNull()
    ).count()
)

missing_ids = (
    df.filter(
        F.col("resource_id").isNull()
    ).count()
)

duplicate_ids = (
    n_in - df.select("resource_id").distinct().count()
)

print("Bronze input rows:", n_in)
print("Expected Raw rows:", raw_rows)
print("Missing extraction timestamps:", missing_timestamps)
print(f"Missing {resource_name} IDs:", missing_ids)
print(f"Duplicate {resource_name} IDs:", duplicate_ids)

if n_in != raw_rows:
    raise RuntimeError(
        "Bronze input count does not match Raw count."
    )

if missing_timestamps > 0:
    raise RuntimeError(
        "Some extraction timestamps are missing."
    )

if missing_ids > 0 or duplicate_ids > 0:
    raise RuntimeError(
        f"{resource_name} ID validation failed."
    )

print(f"{resource_name} Bronze input validation successful.")


StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 71, Finished, Available, Finished, False)

Bronze input rows: 989
Expected Raw rows: 989
Missing extraction timestamps: 0
Missing Condition IDs: 0
Duplicate Condition IDs: 0
Condition Bronze input validation successful.


In [70]:

# CELL 13: CHECK EXISTING BRONZE RUN

if spark.catalog.tableExists(target):

    existing_run = (
        spark.table(target)
        .filter(
            (F.col("run_id") == run_id) &
            (F.col("ingest_date") == load_date)
        )
    )

    existing_run_count = existing_run.count()

    print("Bronze table exists:", target)
    print("Current run ID:", run_id)
    print("Existing records for this run:", existing_run_count)

    if existing_run_count == 0:
        print("This run is not yet loaded. Safe to proceed.")

    elif existing_run_count == raw_rows:
        print("This run is already fully loaded. Skip writing.")

    else:
        raise RuntimeError(
            "Partial Bronze load detected. Stop and investigate."
        )

else:

    existing_run_count = 0

    print("Bronze table does not exist:", target)
    print("A new Bronze table will be created in Cell 14.")


StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 72, Finished, Available, Finished, False)

Bronze table exists: bronze.condition
Current run ID: 660f0af9-ac1a-48f0-812b-989c7c3ba0c1
Existing records for this run: 0
This run is not yet loaded. Safe to proceed.


In [71]:

# CELL 14: APPEND DATA TO BRONZE

if existing_run_count == 0:

    if spark.catalog.tableExists(target):

        existing_columns = spark.table(target).columns

        missing_columns = [
            col for col in existing_columns
            if col not in df.columns
        ]

        extra_columns = [
            col for col in df.columns
            if col not in existing_columns
        ]

        if missing_columns or extra_columns:
            raise RuntimeError(
                f"Schema mismatch. Missing: {missing_columns}, "
                f"Extra: {extra_columns}"
            )

        df_to_write = df.select(*existing_columns)

        (
            df_to_write.write
            .format("delta")
            .mode("append")
            .saveAsTable(target)
        )

    else:

        (
            df.write
            .format("delta")
            .mode("errorifexists")
            .saveAsTable(target)
        )

    print(f"{resource_name} records appended successfully.")

else:

    print("Write skipped because this run is already loaded.")


StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 73, Finished, Available, Finished, False)

Condition records appended successfully.


In [72]:

# CELL 15: VALIDATE BRONZE DATA

bronze_df = spark.table(target)

total_bronze = bronze_df.count()

current_run = bronze_df.filter(
    (F.col("run_id") == run_id) &
    (F.col("ingest_date") == load_date)
)

current_run_count = current_run.count()

distinct_ids = (
    current_run.select("resource_id")
    .distinct()
    .count()
)

print(f"Total Bronze {resource_name} records:", total_bronze)
print("Current run records:", current_run_count)
print("Expected records:", raw_rows)
print("Distinct resource IDs:", distinct_ids)

if current_run_count != raw_rows:
    raise RuntimeError("Bronze record count mismatch.")

if distinct_ids != raw_rows:
    raise RuntimeError("Duplicate resource IDs detected.")

print(f"{resource_name} Bronze validation successful.")

display(
    current_run.select(
        "resource_id",
        "source_last_updated",
        "ingest_date",
        "run_id"
    ).limit(10)
)


StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 74, Finished, Available, Finished, False)

Total Bronze Condition records: 2600
Current run records: 989
Expected records: 989
Distinct resource IDs: 989
Condition Bronze validation successful.


SynapseWidget(Synapse.DataFrame, 0b81f220-f19b-4ab7-b9ba-96f185a79639)

In [73]:
# ============================================
# CELL 16: CHECK BRONZE PIPELINE LOG
# ============================================

bronze_log_check = (
    spark.table("meta.pipeline_log")
    .filter(
        (F.col("run_id") == run_id) &
        (F.col("resource_name") == resource_name) &
        (F.col("load_date") == load_date) &
        (F.col("layer") == "bronze") &
        (F.col("step") == "summary")
    )
)

print("Existing Bronze summary logs:",
      bronze_log_check.count())

display(bronze_log_check)


StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 75, Finished, Available, Finished, False)

Existing Bronze summary logs: 0


SynapseWidget(Synapse.DataFrame, 796ad62c-3131-4711-9725-6a8f9bda3f08)

In [74]:

# CELL 17: LOG BRONZE SUCCESS

if bronze_log_check.count() == 0:

    log_summary(
        layer="bronze",
        fetched=raw_rows,
        written=current_run_count,
        status="SUCCESS",
        message=f"{resource_name} data successfully loaded and validated in Bronze.",
        api_total=raw_rows,
        path=raw_dir
    )

    print(f"{resource_name} Bronze success logged.")

else:

    print(
        f"{resource_name} Bronze success log already exists. "
        "Skipping duplicate log."
    )


StatementMeta(, ebd27adb-ded6-4991-8da2-11bd42f317b3, 76, Finished, Available, Finished, False)

Condition Bronze success logged.
